<a href="https://colab.research.google.com/github/NeoByteVoyager/PytorchLearning/blob/main/ML/MINIST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 加载数据

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader



# 数据转换方法
transform = transforms.Compose([
    transforms.ToTensor(),              # 先把图片转换为tensor
    transforms.Normalize((0.1307,),(0.3081,))   # 再将每个像素点转换为0~1之间得数字
])

# 下载数据集
train_dataset =  torchvision.datasets.MNIST(
    root="./dataset",
    train=True,
    transform=transform,
    download=True
)
test_dataset = torchvision.datasets.MNIST(
    root="./dataset",
    train=False,
    transform=transform,
    download=True
)

# 加载数据集
train_loader = DataLoader(dataset=train_dataset, batch_size=64, shuffle=True,num_workers=2)
test_loader = DataLoader(dataset=test_dataset, batch_size=64, shuffle=False,num_workers=2)


## 检测并定义设备

In [ ]:
# 如果 GPU 可用就用 GPU（cuda），否则用 CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"当前使用的设备: {device}")

当前使用的设备: cuda


## 创建模型

In [ ]:
# 搭建模型
class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784,512),
            nn.ReLU(),
            nn.Linear(512,256),
            nn.ReLU(),
            nn.Linear(256,128),
            nn.ReLU(),
            nn.Linear(128,10)
        )
    def forward(self,x):
        return self.network(x)

model = Model()
model = model.to(device)

## 损失函数和优化器

In [ ]:
# 创建损失函数和优化器
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=1e-4)

## 训练函数

In [ ]:
# 训练函数
def train():
    total_loss = 0
    for image,labels in train_loader:
        image = image.to(device)
        labels = labels.to(device)
        # 前向传播
        y_hat = model(image)
        loss = criterion(y_hat,labels)
        # 反向传播
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss/len(train_loader)

## 测试函数

In [ ]:
def test():
    total = 0
    correct = 0
    with torch.no_grad():
        for image,labels in test_loader:
            image = image.to(device)
            labels = labels.to(device)

            y_hat = model(image)

            _,preds = torch.max(y_hat,dim=1)

            total += labels.shape[0]
            correct += (preds == labels).sum().item()
    print(f"准确率：{correct/total:.2f}%")

## 训练循环

In [ ]:
for epoch in range(10):
    loss = train()
    print(f"epoch:{epoch}|loss:{loss:.4f}")

    test()
